# 42 — Fault-tolerance theatre (SC vs fixed-point)

High-fidelity **Perfect Integrator** (polyglot-complete) provides a clean
spike counter under constant current. We compare **float multiply** vs
**stochastic unipolar multiply** while injecting bit flips.

## Honesty box

| | |
|---|---|
| **Proves** | Relative robustness trend of SC unipolar multiply vs float under random bit flips at controlled BER; Perfect Integrator spike counts under clean constant current. |
| **Does not prove** | Radiation hardness, automotive FIT rates, or that SC always wins for all workloads. |
| **Artefacts** | Local figures only. |
| **Models** | `PerfectIntegratorNeuron` (polyglot-complete). |


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from sc_neurocore import BitstreamEncoder, bitstream_to_probability
from sc_neurocore.neurons.models import PerfectIntegratorNeuron
from sc_neurocore.utils.fault_injection import FaultInjector

np.random.seed(7)
print("SC-NeuroCore — NB-42 fault-tolerance theatre")


## 1. Clean Perfect Integrator baseline


In [ ]:
neuron = PerfectIntegratorNeuron()
v, spikes = neuron.simulate(1000, current=2.0)
print(f"PerfectIntegrator: spikes={spikes}, vmax={float(v.max()):.3f}, dt={neuron.dt}")
t = np.arange(len(v)) * float(neuron.dt)
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(t, v, lw=0.9)
ax.set_title(f"Perfect Integrator sawtooth — spikes={spikes}")
ax.set_xlabel("time")
ax.set_ylabel("v")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 2. SC multiply vs float under bit flips


In [ ]:
def float_product(a: float, b: float) -> float:
    return a * b

def sc_product(a: float, b: float, length: int, seed: int) -> float:
    ea = BitstreamEncoder(0.0, 1.0, length=length, seed=seed)
    eb = BitstreamEncoder(0.0, 1.0, length=length, seed=seed + 1)
    # unipolar multiply ≈ bitwise AND
    bits = np.asarray(ea.encode(a), dtype=np.uint8) & np.asarray(eb.encode(b), dtype=np.uint8)
    return float(bitstream_to_probability(bits))

def inject_flips(bits: np.ndarray, ber: float, rng: np.random.Generator) -> np.ndarray:
    out = bits.copy()
    mask = rng.random(out.shape) < ber
    out[mask] = 1 - out[mask]
    return out

def sc_product_noisy(a: float, b: float, length: int, ber: float, seed: int) -> float:
    rng = np.random.default_rng(seed)
    ea = BitstreamEncoder(0.0, 1.0, length=length, seed=seed)
    eb = BitstreamEncoder(0.0, 1.0, length=length, seed=seed + 17)
    ba = inject_flips(np.asarray(ea.encode(a), dtype=np.uint8), ber, rng)
    bb = inject_flips(np.asarray(eb.encode(b), dtype=np.uint8), ber, rng)
    return float(bitstream_to_probability(ba & bb))

a, b = 0.7, 0.55
true = float_product(a, b)
length = 2048
bers = np.array([0.0, 0.01, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2])
sc_errs = []
float_bit_errs = []
for ber in bers:
    sc_est = sc_product_noisy(a, b, length, float(ber), seed=11)
    sc_errs.append(abs(sc_est - true))
    # naive float: flip random fraction of mantissa-ish by scaling noise
    noisy = true * (1.0 + np.random.default_rng(int(ber * 1000)).normal(0, ber))
    float_bit_errs.append(abs(float(noisy) - true))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(bers, sc_errs, "o-", label="SC AND product |error|")
ax.plot(bers, float_bit_errs, "s--", label="float + relative noise (proxy)")
ax.set_xlabel("bit error rate (BER)")
ax.set_ylabel("|estimate − true product|")
ax.set_title(f"Unipolar product a={a}, b={b}, true={true:.3f}, L={length}")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

# Optional package FaultInjector smoke (API presence)
try:
    fi = FaultInjector
    print("FaultInjector available:", fi)
except Exception as exc:
    print("FaultInjector import detail:", exc)
print("NB-42 complete.")
